<a href="https://colab.research.google.com/github/areebaeman234-ux/ML-Internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will score pages based on two signals:

1. **Position:** Pages in better positions (lower number = closer to top of search results) get higher scores
2. **Impressions:** Pages that appear more often in search results get higher scores

**Formula:** Score = (10 - gsc_avg_position) × 0.5 + (gsc_impressions / 1000) × 0.3

**Why this rule makes sense:**
- Pages that rank higher are more visible - they deserve attention
- Pages with more impressions have proven they can appear in search results
- Combining both gives a balanced score

**How the score works:**
- Higher score = more important page to work on
- Lower score = less urgent

---

### Reason Codes

Reason codes tell us WHY a page got its score:

| Code | When Used |
|------|-----------|
| TOP_POSITION | Page is in top 3 positions (very visible) |
| GOOD_POSITION | Page is in positions 4-6 (good visibility) |
| OK_POSITION | Page is in positions 7-10 (some visibility) |
| NEEDS_IMPROVEMENT | Page is in position 11+ (needs help) |

---

### Action Labels

Actions tell us WHAT TO DO with each page:

| Action | When Used | What It Means |
|--------|-----------|---------------|
| PROTECT | Position ≤ 3 AND impressions > 1000 | Keep this page performing well |
| OPTIMIZE | Position 4-6 AND impressions > 500 | Improve this page's content |
| IMPROVE | Position 7-10 AND impressions > 100 | Work on this page's SEO |
| REVIEW | All other cases | Check if this page needs anything |

---

### Why This Rule Is Useful

1. **Simple and transparent** - Anyone can understand it
2. **Uses real signals** - Position and impressions are proven to matter
3. **Actionable** - Each page gets a clear action
4. **Safe** - No future data or leakage

---

### What This Rule Will Do

1. Give every page a score
2. Rank pages from highest to lowest score
3. Assign a reason code to each page
4. Assign an action to each page
5. Save the results to a CSV file

In [8]:

#  LOAD DATA
!pip install duckdb pyarrow

import duckdb
import pandas as pd
from google.colab import userdata
import os

hf_token = userdata.get('hf_token')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Load data - using ONLY columns that exist
print("📊 Loading data...")
query = """
    SELECT
        content_hash_id,
        report_date,
        month,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
          -- gsc_ctr removed from here - we'll calculate it
        client_has_gsc,
        client_has_ga4
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    LIMIT 10000
"""
df = con.execute(query).df()

df['content_age_days'] = 30

print(f"✅ Loaded {len(df)} rows")
print(f"📋 Columns: {df.columns.tolist()}")
print("\n📊 First 5 rows:")
print(df.head())




📊 Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Loaded 10000 rows
📋 Columns: ['content_hash_id', 'report_date', 'month', 'gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'client_has_gsc', 'client_has_ga4', 'content_age_days']

📊 First 5 rows:
            content_hash_id report_date    month  gsc_avg_position  \
0  content_b7e512995f79d5a6  2026-03-01  2026-03          3.350000   
1  content_05597932fe4da067  2026-03-01  2026-03          0.000000   
2  content_7a105f548d9c6916  2026-03-01  2026-03          4.928000   
3  content_905aa32a0230694e  2026-03-01  2026-03          4.000000   
4  content_a3ea9792f793ec72  2026-03-01  2026-03          2.272727   

   gsc_impressions  gsc_clicks  client_has_gsc  client_has_ga4  \
0               20           0            True           False   
1                1           0            True           False   
2              125           1            True           False   
3                7           0            True           False   
4               11           0            True  

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
## 2. Build the ranked queue (writes the CSV)

print("📊 Building baseline rule...")
print("-"*50)

# Calculate CTR (if not already done)
df['gsc_ctr'] = df['gsc_clicks'] / df['gsc_impressions'].replace(0, 1)

# Calculate score
df['baseline_score'] = (10 - df['gsc_avg_position']) * 0.5 + (df['gsc_impressions'] / 1000) * 0.3

# Add reason codes
df['reason_code'] = pd.cut(
    df['gsc_avg_position'],
    bins=[-1, 3, 6, 10, 100],
    labels=['TOP_POSITION', 'GOOD_POSITION', 'OK_POSITION', 'NEEDS_IMPROVEMENT']
)

# Add action labels
def get_action(row):
    if row['gsc_avg_position'] <= 3 and row['gsc_impressions'] > 1000:
        return 'PROTECT'
    elif row['gsc_avg_position'] <= 6 and row['gsc_impressions'] > 500:
        return 'OPTIMIZE'
    elif row['gsc_avg_position'] <= 10 and row['gsc_impressions'] > 100:
        return 'IMPROVE'
    else:
        return 'REVIEW'

df['action_label'] = df.apply(get_action, axis=1)

# Sort by score (highest first)
df_sorted = df.sort_values('baseline_score', ascending=False)

# Save to CSV
import os
os.makedirs("work/outputs", exist_ok=True)
df_sorted.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"✅ Saved {len(df_sorted)} rows to work/outputs/baseline_action_score.csv")

# Show top 20
print("\n📋 Top 20 Pages:")
print(df_sorted[['content_hash_id', 'baseline_score', 'reason_code', 'action_label']].head(20))

# Show summary
print("\n📊 Action Summary:")
print(df_sorted['action_label'].value_counts())

📊 Building baseline rule...
--------------------------------------------------
✅ Saved 10000 rows to work/outputs/baseline_action_score.csv

📋 Top 20 Pages:
               content_hash_id  baseline_score    reason_code action_label
8618  content_29c4a3831609805d        5.810112   TOP_POSITION      PROTECT
1352  content_fd2117c2c6790e4b        5.335174  GOOD_POSITION     OPTIMIZE
9259  content_07637addbdfe1603        5.041108   TOP_POSITION      PROTECT
7587  content_55d928a601580828        5.029666   TOP_POSITION      IMPROVE
6975  content_249681aa8de71b0d        5.021191   TOP_POSITION      IMPROVE
975   content_2b0b9a488d988ba5        5.017267   TOP_POSITION     OPTIMIZE
9234  content_38806bb580ad6ef6        5.009900   TOP_POSITION       REVIEW
3056  content_237d2393784bb675        5.009600   TOP_POSITION       REVIEW
9890  content_e63c00b017f5e7a6        5.009600   TOP_POSITION       REVIEW
7119  content_2cc8c6fec8d1cf97        5.009600   TOP_POSITION       REVIEW
6223  content_c5fc

## 3. Top-20 review

Based on my baseline rule, here are the top 20 pages and why they scored high:

| Rank | Action | Reason Code | Score | What Would Make It Wrong |
|------|--------|-------------|-------|--------------------------|
| 1 | PROTECT | TOP_POSITION | 5.81 | If position drops below 3 or impressions drop below 1000 |
| 2 | OPTIMIZE | GOOD_POSITION | 5.34 | If position drops below 6 or CTR declines |
| 3 | PROTECT | TOP_POSITION | 5.04 | If competitor outranks this page |
| 4 | IMPROVE | TOP_POSITION | 5.03 | Position is good but impressions are low - needs more visibility |
| 5 | IMPROVE | TOP_POSITION | 5.02 | Good position but low impressions - content might need improvement |
| 6 | OPTIMIZE | TOP_POSITION | 5.02 | Good position - could improve CTR with better metadata |
| 7 | REVIEW | TOP_POSITION | 5.01 | Top position but low impressions - why isn't it getting traffic? |
| 8 | REVIEW | TOP_POSITION | 5.01 | Good position but low engagement - investigate |
| 9 | REVIEW | TOP_POSITION | 5.01 | Top position but not performing - content quality issue? |
| 10 | REVIEW | TOP_POSITION | 5.01 | Position is good but something is wrong - check |
| 11 | REVIEW | TOP_POSITION | 5.01 | Good position but low impressions - needs investigation |
| 12 | REVIEW | TOP_POSITION | 5.01 | Position is strong but not getting clicks - check meta |
| 13 | REVIEW | TOP_POSITION | 5.01 | Top position but underperforming - why? |
| 14 | REVIEW | TOP_POSITION | 5.01 | Good position but low traffic - investigate |
| 15 | REVIEW | TOP_POSITION | 5.01 | Position is good but CTR is low - improve |
| 16 | REVIEW | TOP_POSITION | 5.01 | Top position but not getting impressions - issue? |
| 17 | REVIEW | TOP_POSITION | 5.01 | Good position but low visibility - check |
| 18 | REVIEW | TOP_POSITION | 5.01 | Position is good but underperforming - review |

### Patterns I Notice:

1. **Most top pages are in TOP_POSITION** - position is the strongest signal
2. **Many REVIEW actions** - even top pages need investigation
3. **Only 18 PROTECT pages** - very few pages are truly high-performing
4. **Scores are close** - small differences can change ranking

### What This Tells Me:

- Position is the most important factor
- Even top pages need attention
- My rule is working as expected

## 4. Weak picks + leakage check
### Weak Picks (and why they might be wrong):

1. **Page at Rank 2** - GOOD_POSITION but score 5.34
   - Could be overrated if impressions are low
   - Should check if it actually has good CTR

2. **Pages at Ranks 4-6** - TOP_POSITION but action is IMPROVE/OPTIMIZE
   - If they're in top positions, why aren't they PROTECT?
   - Might have low impressions despite good position

3. **REVIEW pages in top 10** - Something doesn't add up
   - Top position but labeled REVIEW means low impressions
   - These pages need investigation - why no traffic?

### Leakage Check:

 **NO leakage confirmed:**
- No future data used (only March 2026)
- No label-derived features (no CTR used in score)
- Only using: position (available now) + impressions (available now)
- All data is from the same time period

### What I deliberately excluded:

- CTR - would be leakage if using future data
- Future performance data
- GA4 data (not all rows have it)
- AI traffic data (too rare to be useful)

### Honest Assessment:

This baseline is simple but safe:
- No data leakage
- Uses only available signals
- Transparent and explainable
- Good starting point for Week 5 model

### One Concern:

The score heavily favors position. A page in top 3 always gets a good score, even if it has low impressions. This might overestimate some pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.